# LLM Zoomcamp 2026 — Homework 05: Monitoring

This notebook instruments the course RAG pipeline with OpenTelemetry, captures token and latency metrics, stores spans in SQLite, and derives the answers to Questions 1–6.

## Sources

The implementation follows the official 2026 homework and the monitoring module:

- [Homework 05 — Monitoring](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md)
- [Module 05 — Monitoring](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/05-monitoring)
- [Capturing Metrics lesson](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/05-monitoring/lessons/04-metrics.md)
- [Database lesson](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/05-monitoring/lessons/05-database.md)
- [Querying lesson](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/05-monitoring/lessons/06-querying.md)


A reference run using the official starter, commit `8c1834d`, and `gpt-5.4-mini` produced approximately **7,111 input tokens**, an LLM span of about **4.7 seconds**, and the same input-token count in all four repeated calls.

## 1. Environment

In [ ]:
%pip install -q gitsource minsearch openai python-dotenv pandas opentelemetry-api opentelemetry-sdk

In [ ]:
import os
import sqlite3
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from dotenv import load_dotenv
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not found in the environment.")

QUERY = "How does the agentic loop keep calling the model until it stops?"

The starter files are downloaded from the course repository only when they are not already present.

In [ ]:
BASE_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/"
    "main/cohorts/2026/05-monitoring"
)

for filename in ("rag_helper.py", "starter.py"):
    path = Path(filename)
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{filename}", path)

from rag_helper import RAGBase
from starter import rag, documents

print(f"Model: {rag.model}")
print(f"Indexed documents: {len(documents)}")

## 2. OpenTelemetry instrumentation

The tracer is passed explicitly to the RAG subclass. This keeps the console and SQLite exporters separate while allowing the entire notebook to run in one kernel.

In [ ]:
def calculate_cost(model, usage):
    if "gpt-5.4-mini" in model:
        return (
            usage.input_tokens * 0.15
            + usage.output_tokens * 0.60
        ) / 1_000_000
    return 0.0


class CapturingExporter(SpanExporter):
    def __init__(self):
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass

    def force_flush(self, timeout_millis=30_000):
        return True


class RAGTraced(RAGBase):
    def __init__(self, *args, tracer, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    def rag(self, query):
        with self.tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search"):
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            cost = calculate_cost(self.model, usage)

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("total_tokens", usage.total_tokens)
            span.set_attribute("cost", cost)

            return response

## 3. Questions 1–3: console trace, tokens, and timing

In [ ]:
capture_exporter = CapturingExporter()

console_provider = TracerProvider()
console_provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
console_provider.add_span_processor(
    SimpleSpanProcessor(capture_exporter)
)
console_tracer = console_provider.get_tracer("llm-zoomcamp")

rag_console = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    model=rag.model,
    tracer=console_tracer,
)

capture_exporter.spans.clear()
answer = rag_console.rag(QUERY)
print(answer)

### Q1. First trace

The call is wrapped by three spans: `rag`, `search`, and `llm`.

In [ ]:
span_names = [span.name for span in capture_exporter.spans]
q1_answer = len(span_names)

print("Spans:", span_names)
print("Q1 answer:", q1_answer)

### Q2. Capturing metrics as span attributes

The closest official option is selected from the measured `input_tokens` value.

In [ ]:
llm_span = next(
    span for span in capture_exporter.spans if span.name == "llm"
)
input_tokens = llm_span.attributes["input_tokens"]

q2_options = [700, 7_000, 70_000, 700_000]
q2_answer = min(
    q2_options,
    key=lambda option: abs(option - input_tokens),
)

print("Measured input tokens:", input_tokens)
print("Q2 answer:", q2_answer)

### Q3. Span timing

OpenTelemetry timestamps are stored in nanoseconds.

In [ ]:
llm_duration_ms = (
    llm_span.end_time - llm_span.start_time
) / 1_000_000

if llm_duration_ms < 100:
    q3_answer = "Under 100ms"
elif llm_duration_ms < 500:
    q3_answer = "100-500ms"
elif llm_duration_ms < 2_000:
    q3_answer = "500-2000ms"
else:
    q3_answer = "Over 2000ms"

print(f"LLM duration: {llm_duration_ms:,.1f} ms")
print("Q3 answer:", q3_answer)

## 4. SQLite span exporter

In [ ]:
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute(
            '''
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
            '''
        )
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attributes = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attributes.get("input_tokens"),
                    attributes.get("output_tokens"),
                    attributes.get("cost"),
                ),
            )

        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30_000):
        return True

Four identical RAG calls are stored in a fresh database. This single run provides the data needed for Questions 4–6.

In [ ]:
DB_PATH = Path("traces.db")
DB_PATH.unlink(missing_ok=True)

sqlite_exporter = SQLiteSpanExporter(DB_PATH)
sqlite_provider = TracerProvider()
sqlite_provider.add_span_processor(
    SimpleSpanProcessor(sqlite_exporter)
)
sqlite_tracer = sqlite_provider.get_tracer("llm-zoomcamp")

rag_sqlite = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    model=rag.model,
    tracer=sqlite_tracer,
)

for run_number in range(1, 5):
    rag_sqlite.rag(QUERY)
    print(f"Completed run {run_number}")

sqlite_provider.force_flush()

In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    spans_df = pd.read_sql_query(
        "SELECT * FROM spans ORDER BY start_time",
        connection,
    )

spans_df["duration_ms"] = (
    spans_df["end_time"] - spans_df["start_time"]
) / 1_000_000

spans_df

### Q4. Saving traces to SQLite

In [ ]:
saved_span_names = set(spans_df["name"])

if saved_span_names == {"rag"}:
    q4_answer = "Only rag"
elif saved_span_names == {"rag", "llm"}:
    q4_answer = "rag and llm"
elif saved_span_names == {"rag", "search", "llm"}:
    q4_answer = "rag, search, and llm"
else:
    q4_answer = "search, llm, and judge"

print("Saved span names:", sorted(saved_span_names))
print("Q4 answer:", q4_answer)

### Q5. Querying trace data

The parent `rag` spans are excluded before aggregating duration by child span.

In [ ]:
duration_by_span = (
    spans_df.loc[spans_df["name"] != "rag"]
    .groupby("name", as_index=False)["duration_ms"]
    .sum()
    .sort_values("duration_ms", ascending=False)
)

q5_answer = duration_by_span.iloc[0]["name"]

print(duration_by_span.to_string(index=False))
print("Q5 answer:", q5_answer)

### Q6. Token stability across runs

In [ ]:
token_runs = (
    spans_df.loc[spans_df["name"] == "llm", "input_tokens"]
    .dropna()
    .astype(int)
    .reset_index(drop=True)
)

relative_range = (
    token_runs.max() - token_runs.min()
) / token_runs.mean()

if token_runs.nunique() == 1:
    q6_answer = "They're identical"
elif relative_range <= 0.10:
    q6_answer = "Within 10% of each other"
elif relative_range <= 0.50:
    q6_answer = "Within 50% of each other"
else:
    q6_answer = "They vary more than 50%"

print("Input tokens by run:", token_runs.tolist())
print(f"Relative range: {relative_range:.2%}")
print("Q6 answer:", q6_answer)

## 5. Final answers

In [ ]:
final_answers = pd.DataFrame(
    {
        "Question": ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"],
        "Answer": [
            str(q1_answer),
            str(q2_answer),
            q3_answer,
            q4_answer,
            q5_answer,
            q6_answer,
        ],
    }
)

final_answers